---
title: Structural Variants Detected
subtitle: Putative structural variants as detected by PLACEHOLDER
date: "9999-12-31"
edit_url: null
---

In [ ]:
from harpy.report.by_chromosome import sv_by_chromosome
from harpy.report.components import colored_boxes, print_html, standard_itable
from harpy.report.theme import sv_colors
import itables
import pandas as pd
from pathlib import Path
import sys
itables.init_notebook_mode(connected=True)

In [12]:
faidx = "genome.fasta.fai"
statsdir = "."
contigs = "default"

In [ ]:
sv = pd.concat([pd.read_table(i) for i in Path(statsdir).glob("*.bedpe")]).reset_index(drop = True)
if len(sv) < 1:
    print_html("No structural variants were detected.")
    sys.exit(0)

In [ ]:
def process_variants(df, bin_size=50):
    """
    Group variants by binning positions into windows
    """
    # Create binned positions
    df['start_bin'] = (df['position_start'] // bin_size) * bin_size
    df['end_bin'] = (df['position_end'] // bin_size) * bin_size

    # Group by contig, type, and binned positions
    grouped = df.groupby(['contig', 'type', 'start_bin', 'end_bin']).agg(
        position_start=('position_start', 'median'),
        position_end=('position_end', 'median'),
        n_samples=('sample', 'count'),
        samples=('sample', list)
    ).reset_index(drop=False)

    # Clean up
    grouped['position_start'] = grouped['position_start'].astype(int)
    grouped['position_end'] = grouped['position_end'].astype(int)
    grouped = grouped[['contig', 'position_start', 'position_end', 'type', 'n_samples', 'samples']]

    return grouped

In [15]:
_sv_types = {"INV": "Inversion", "DEL": "Deletion", "DUP": "Duplication", "BND": "Breakend"}

unique_variants = process_variants(sv, bin_size=100)
for k,v in _sv_types.items():
    unique_variants['type'] = unique_variants['type'].str.replace(k,v)

unique_variants.insert(
    3,
    "var_length",
    unique_variants['position_end'] - unique_variants['position_start']
)
unique_variants['samples'] = unique_variants['samples'].str.join(', ')

In [16]:
_sv_counts = unique_variants['type'].value_counts().to_dict()
boxes = colored_boxes()
for k,v in _sv_types.items():
    boxes.add(_sv_counts.get(v, 0), v + "s", sv_colors(k))

boxes.render()

## Variants Identified
This report details the structural variants identified by [LEVIATHAN](https://github.com/morispi/LEVIATHAN)
[](https://doi.org/10.1101/2021.03.25.437002). Below are two tables, one with the unique putative structural variants, and the other with all variant detections. 

::::{tab-set}
:::{tab-item} Unique Variants
This table details the overview of variants `LEVIATHAN` detected when consolidating variant calls of the same type within a 100bp window. For example, if two deletions on the same contig had their breakpoints within 20bp, the deletions were merged[^1] and considered
a duplicate of each other. The `n_samples` column is the number of samples that putatively have this variant, whereas `samples` is a list of those samples.
![](#unique_sv_table)
:::
:::{tab-item} All Variants
This table details all the variants `LEVIATHAN` detected and evidence for those detections.
![](#all_sv_table)
:::
::::

[^1]:
    The merging is only for reporting/summarizing purposes, the source data was not modified.

In [17]:
#| label: unique_sv_table

standard_itable(unique_variants, "sv.unique", fixedcols = 3)

Loading ITables v2.6.2 from the internet... (need help?)


In [18]:
#| label: all_sv_table

standard_itable(sv.drop(columns = ['start_bin', "end_bin"]), "sv.all", fixedcols = 1)

Loading ITables v2.6.2 from the internet... (need help?)


In [19]:
contig_lens = pd.read_table(faidx, header = None).drop(columns = [2,3,4]).rename(columns = {0 : 'contig', 1 : 'length'})

if contigs == "default":
    _keep = contig_lens.nlargest(30, 'length')['contig'].tolist()
else:
    _keep = contigs

unique_variants = unique_variants[unique_variants['contig'].isin(_keep)].reset_index(drop = True)
contig_lens = contig_lens[contig_lens['contig'].isin(_keep)].reset_index(drop = True)

unique_variants = unique_variants.merge(contig_lens, on = 'contig')

In [20]:
sv_by_chromosome(unique_variants, "Structural Variants Detected")

alt.Chart(...)

## Supporting Info

:::{dropdown} Variant Types
:closed:

**Inversion (INV)**: A segment that broke off and reattached within the same chromosome, but in reverse orientation

**Duplication (DUP)**: A type of mutation that involves the production of one or more copies of a gene or region of a chromosome

**Deletion (DEL)**: A type of mutation that involves the loss of one or more nucleotides from a segment of DNA

**Breakend (BND)**: The endpoint of a structural variant, which can be useful to explain complex variants
:::